<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Hierarchies that must add up

**[Hierarchies that must add up](https://learning.nextia-ai.com/courses/forecasting/m06/hierarchies-that-must-add-up/)** · Forecasting and Time Series: Predict What Comes Next · Module 6, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build Larkfield's hierarchy (total, region, category, product), see that separate forecasts do not add up, and reconcile them bottom-up, top-down and with MinT.

| | |
|---|---|
| **Time** | About 40 minutes, of which about 1 minute of waiting for the forecasts |
| **Needs** | An internet connection for the setup cells. No account and no key. The first cell also installs hierarchicalforecast 1.5.3. |
| **You should know** | ETS, from [Exponential smoothing](https://learning.nextia-ai.com/courses/forecasting/m04/exponential-smoothing/). WAPE, from [Measure the error that matters](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m06/hierarchies-that-must-add-up/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M06-L01-hierarchies-that-must-add-up/hierarchies-that-must-add-up-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, hierarchicalforecast 1.5.3, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 hierarchicalforecast==1.5.3 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)
import hierarchicalforecast; print("hierarchicalforecast", hierarchicalforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Four numbers that do not add up

Kenji forecasts two products in one region: 30 and 50 units. Grace's colleague forecasts the region's total on its own: 90. Which is right? Both can be reasonable, and they still disagree by 10.

**Reconciliation** makes all levels agree. **Bottom-up** adds up the products (80). **Top-down** keeps the total (90) and splits it by the products' shares (33.75 and 56.25). **MinT** (minimum trace) finds the numbers closest to all the forecasts, weighted by how good each level was in the past.

> **Predict.** Bottom-up changes the total. Top-down changes the products. Which one would you trust for a slow product? Then run the cell.

In [ ]:
products, total = np.array([30.0, 50.0]), 90.0
print("bottom-up: products", products.tolist(), "total", products.sum())
print("top-down:  products", (total * products / products.sum()).tolist(), "total", total)

> **Check.** You should see `bottom-up: products [30.0, 50.0] total 80.0` and `top-down:  products [33.75, 56.25] total 90.0`.

## 2. Larkfield's hierarchy

At the bottom are the 600 series of one product in one region. Above them: each category in each region (30), each region (3) and the total (1). hierarchicalforecast's `aggregate()` builds every level and a **summing matrix** `S` that says which bottom series add up to which total. Products that start later get zero sales before their launch, so that every series has every day.

In [ ]:
from hierarchicalforecast.utils import aggregate

d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
products = pd.read_csv("data/products.csv", parse_dates=["launch_date"])
grid = pd.MultiIndex.from_product([pd.date_range(d["date"].min(), d["date"].max()), sorted(d["region"].unique()),
                                   sorted(d["product_id"].unique())], names=["ds", "region", "product"])
bottom = d.rename(columns={"date": "ds", "product_id": "product", "sales": "y"}).set_index(["ds", "region", "product"])["y"]
bottom = bottom.reindex(grid).reset_index()
bottom = bottom.merge(products.rename(columns={"product_id": "product"})[["product", "category", "launch_date"]], on="product")
bottom.loc[bottom["ds"] < bottom["launch_date"], "y"] = 0.0
# the missing day: the mean of the same weekday one week before and after, per series
bottom = bottom.sort_values(["region", "product", "ds"])
g = bottom.groupby(["region", "product"])["y"]
bottom["y"] = bottom["y"].fillna(((g.shift(7) + g.shift(-7)) / 2).round())
bottom["total"] = "total"
spec = [["total"], ["total", "region"], ["total", "region", "category"], ["total", "region", "category", "product"]]
Y, S, tags = aggregate(bottom[["total", "region", "category", "product", "ds", "y"]], spec)
print({k: len(v) for k, v in tags.items()}, "| rows:", len(Y))

> **Check.** You should see `{'total': 1, 'total/region': 3, 'total/region/category': 30, 'total/region/category/product': 600}` and `rows: 461552` (634 series × 728 days).

## 3. Forecast every level, then compare

The next cell forecasts all 634 series with ETS at the course's last cut-off (15 March 2026) for 14 days, and compares the total's own forecast with the sum of the 600 product forecasts.

> **Predict.** Will the total's own forecast be higher or lower than the sum of the products? Then run the cell (about a minute).

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

origin = ORIGINS[-1]
train = Y[Y["ds"] <= origin]
test = Y[(Y["ds"] > origin) & (Y["ds"] <= origin + pd.Timedelta(days=14))]
sf = StatsForecast(models=[AutoETS(season_length=7)], freq="D")
base = sf.forecast(df=train, h=14, fitted=True)
fitted = sf.forecast_fitted_values()
own_total = base[base["unique_id"] == "total"]["AutoETS"].sum()
sum_products = base[base["unique_id"].isin(tags["total/region/category/product"])]["AutoETS"].sum()
print(f"total's own forecast {own_total:.0f} | sum of the products {sum_products:.0f} | gap {100 * (own_total - sum_products) / sum_products:+.1f}%")

> **Check.** You should see `total's own forecast 14346 | sum of the products 14764 | gap -2.8%`.

Two honest forecasts of the same thing disagree by several percent. Kenji orders per product; the finance team plans with the total. Without reconciliation, they plan with different numbers.

## 4. Reconcile

`HierarchicalReconciliation` takes the base forecasts, the in-sample fitted values (MinT uses their errors) and the summing matrix, and returns coherent forecasts for every method.

In [ ]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace, TopDown

hrec = HierarchicalReconciliation(reconcilers=[BottomUp(), TopDown(method="forecast_proportions"),
                                               MinTrace(method="mint_shrink", nonnegative=True)])
rec = hrec.reconcile(Y_hat_df=base, Y_df=fitted, S_df=S, tags=tags)
columns = {"base (not coherent)": "AutoETS", "bottom-up": "AutoETS/BottomUp",
           "top-down": "AutoETS/TopDown_method-forecast_proportions",
           "MinT": [c for c in rec.columns if "MinTrace" in c][0]}
m = rec.merge(test, on=["unique_id", "ds"])
result = {}
for name, col in columns.items():
    result[name] = {level.split("/")[-1]: round(float(100 * (m[m["unique_id"].isin(ids)]["y"] - m[m["unique_id"].isin(ids)][col]).abs().sum()
                                                       / m[m["unique_id"].isin(ids)]["y"].sum()), 1)
                    for level, ids in tags.items()}
print("WAPE % by level, last cut-off:")
print(pd.DataFrame(result).T.to_string())

> **Check.** You should see four rows and four levels (`category` is each category in one region). The base row is 14.2, 14.0, 28.1 and 62.9; MinT is 12.3, 12.5, 28.2 and 61.3.

At this cut-off, MinT is the best or close to the best at every level, and the only method that improves the products (61.3% against 62.9%). Bottom-up is best for the total (12.2%) but worst for the categories (30.7%). Top-down is no better than the base forecasts. Every reconciled column adds up; the base column does not.

## 5. Your turn: do they add up now?

> **Your turn.** For the MinT column (`columns["MinT"]`), compute the gap in % between the total's forecast and the sum of the 600 product forecasts over the 14 days, as in section 3, rounded to 1 decimal. Put it in `mint_gap` as a Python float. Then run the check cell.

In [ ]:
mint_gap = ...  # the gap in % after MinT

In [ ]:
expect('mint_gap', mint_gap, 0.0, tolerance=0.05)

After reconciliation, the gap is 0: every level tells the same story. MinT with `nonnegative=True` also keeps every product at zero or above.

## 6. Failure case: one cut-off is not a verdict

The table above is one cut-off. The course measured the same comparison at the last 4 cut-offs (`reference` numbers on the lesson page): the order of the methods changes from one cut-off to the next at some levels. Before you choose a reconciliation method for Kenji, run the full backtest, level by level, and choose for the level where the decision is made: here, the product in a region.

## Next

You can now make a hierarchy add up. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m06/hierarchies-that-must-add-up/) has the diagrams, the 4-cut-off results and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Intermittent demand](https://learning.nextia-ai.com/courses/forecasting/m06/intermittent-demand/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m06/hierarchies-that-must-add-up/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.